In [1]:
# ===================================================================
# SISTEMA DE ANÁLISIS AMBIENTAL PARA AULA 1.6
# ===================================================================
# AUTOR: Carmen Gómez Alarcón
# FECHA: 2026
# FASE: Paso 1. Ingesta, limpieza y agregación de datos brutos
# DESCRIPCIÓN:
#   Procesa datos de sensores (temperatura, humedad, CO2), condiciones
#   exteriores, ocupación y ventilación para generar reportes semanales
#   y análisis de bloques de ocupación.
#   Los timestamps son los reales de cada dataset (sin regularización);
#   cada fila arrastra su último valor conocido (ffill) hasta el siguiente dato.
# ===================================================================
# UBICACIÓN DE LOS SENSORES EN EL AULA 1.6:
#   - Corridor (Pasillo): Único sensor, junto a la pared, orientado al pasillo.
# ===================================================================

import pandas as pd
import os
import calendar
from datetime import datetime, timedelta, time
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=pd.errors.PerformanceWarning)

# ==================== CONFIGURACIÓN ====================
OUTDOOR_VARIABLES = [
    'Temperature_Outdoor', 'Temp_Outdoor_Increment',
    'Humidity_Outdoor', 'Humidity_Outdoor_Increment',
    'Solar_Radiation', 'Solar_Radiation_Increment'
]

SENSOR_VARIABLES = ['Temperature', 'Temp_Increment', 'Humidity', 'Humidity_Increment', 'CO2', 'CO2_Increment']

sensor_files = [
    ('1.6 sensor I.xlsx', 1.6, 'Corridor')
]


# ==================== FUNCIONES AUXILIARES ====================

def filter_weekdays_time(df, start_time=time(6, 0), end_time=time(23, 30)):
    """Filtra el DataFrame para conservar solo registros de lunes a viernes
    entre las 6:00 y las 23:30, eliminando fines de semana y horario nocturno."""
    if df is None:
        return None
    return df[
        (~df['DateTime'].dt.dayofweek.isin([5, 6])) &
        (df['DateTime'].dt.time >= start_time) &
        (df['DateTime'].dt.time <= end_time)
    ]

def load_dataframe(file_path, date_col_name=None):
    """Carga un archivo Excel y detecta automáticamente la columna de fecha,
    convirtiendo su contenido a tipo datetime. Descarta filas sin fecha válida."""
    if not os.path.exists(file_path):
        return None
    df = pd.read_excel(file_path)
    if date_col_name is None:
        for col in df.columns:
            if 'date' in str(col).lower() or 'fecha' in str(col).lower():
                date_col_name = col
                break
        if date_col_name is None:
            date_col_name = df.columns[0]
    df['DateTime'] = pd.to_datetime(df[date_col_name], errors='coerce', dayfirst=True)
    return df.dropna(subset=['DateTime'])

def load_outdoor_data():
    """Carga los datos climáticos exteriores del archivo SIAR y los renombra
    a los nombres estándar del sistema (temperatura, humedad, radiación solar)."""
    df = load_dataframe('Outdoor parameters. SIAR.xlsx')
    if df is None:
        return None

    result = pd.DataFrame()
    result['DateTime'] = df['DateTime']

    col_map = {}
    for col in df.columns:
        col_lower = str(col).lower()
        if 'temp' in col_lower:
            col_map['Temperature_Outdoor'] = col
        elif 'hr' in col_lower or 'hum' in col_lower:
            col_map['Humidity_Outdoor'] = col
        elif 'solar' in col_lower or 'radiation' in col_lower:
            col_map['Solar_Radiation'] = col

    for var_name, col_name in col_map.items():
        result[var_name] = pd.to_numeric(df[col_name], errors='coerce')

    return result

def save_to_excel(df, file_out, sheet_name, freeze_col='B2'):
    """Guarda un DataFrame en Excel aplicando formato visual a la cabecera."""
    with pd.ExcelWriter(file_out, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        worksheet = writer.sheets[sheet_name[:31]]
        for cell in worksheet[1]:
            cell.fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
            cell.font = Font(bold=True, size=11)
            cell.alignment = Alignment(horizontal='center')
        worksheet.freeze_panes = freeze_col
        for col_idx in range(1, len(df.columns) + 1):
            worksheet.column_dimensions[get_column_letter(col_idx)].width = 22

def split_into_weeks(df, year, month):
    """Divide los datos de un mes en cuatro bloques semanales:
    días 1-7, 8-14, 15-21 y 22-fin de mes."""
    last_day = calendar.monthrange(year, month)[1]
    ranges = [(1, 7), (8, 14), (15, 21), (22, last_day)]
    weeks = []
    for i, (start, end) in enumerate(ranges, 1):
        week_start = datetime(year, month, start)
        week_end   = datetime(year, month, end, 23, 59, 59)
        week_data  = df[(df['DateTime'] >= week_start) & (df['DateTime'] <= week_end)]
        if len(week_data) > 0:
            weeks.append((f'week_{i}_days_{start}-{end}', week_data, week_start, week_end))
    return weeks

def parse_detailed_ventilation(door_value, window_value):
    """Interpreta el estado de puerta y ventana a partir de texto libre."""
    def parse_value(value):
        if pd.isna(value):
            return 0
        return 1 if any(x in str(value).lower().strip() for x in ['abierta', 'abierto']) else 0
    return parse_value(door_value), parse_value(window_value)

def load_auxiliary_data(file_aux):
    """
    Carga el archivo de ocupación y ventilación del aula.
    ✅ FIX 1: La columna 'Date' contiene el serial numérico de Excel (fecha+hora
               combinadas como número decimal). Se convierte correctamente usando
               origin='1899-12-30' en lugar de parsear 'Fecha' (que solo tiene fecha).
    ✅ FIX 2: dropna() antes de astype(int) para evitar IntCastingNaNError.
    """
    if not os.path.exists(file_aux):
        print(f"   File not found: {file_aux}")
        return None, None

    df_aux = pd.read_excel(file_aux)
    print(f"   Loaded: {len(df_aux)} rows")

    # --- 1. Detectar y parsear columna de fecha ---
    # Prioridad: columna llamada exactamente 'Date' (serial Excel con fecha+hora)
    # Fallback: cualquier columna con 'fecha' o 'time' en el nombre
    date_col = None
    for col in df_aux.columns:
        if col.lower() == 'date':
            date_col = col
            break
    if date_col is None:
        for col in df_aux.columns:
            col_lower = col.lower()
            if any(keyword in col_lower for keyword in ['fecha', 'time']):
                date_col = col
                break
    if date_col is None:
        for col in df_aux.columns:
            if df_aux[col].dtype == 'datetime64[ns]':
                date_col = col
                break
    if date_col is None:
        print("   ⚠ No date column found. Skipping auxiliary data.")
        return None, None

    # Convertir: si es numérico (serial Excel con hora decimal), usar origin
    if pd.api.types.is_numeric_dtype(df_aux[date_col]):
        print(f"   Date column '{date_col}' detected as Excel serial → converting with origin")
        df_aux['DateTime'] = pd.to_datetime(
            df_aux[date_col], origin='1899-12-30', unit='D', errors='coerce'
        )
    else:
        df_aux['DateTime'] = pd.to_datetime(df_aux[date_col], errors='coerce', dayfirst=True)

    df_aux = df_aux.dropna(subset=['DateTime'])
    print(f"   DateTime range after parse: {df_aux['DateTime'].min()} → {df_aux['DateTime'].max()}")

    # --- 2. Detectar columna de ocupación ---
    occ_col = None
    occ_keywords = [
        'occupancy', 'ocupación', 'ocupacion', 'presencia', 'presence',
        'estado', 'status', 'ocupado', 'occupied'
    ]
    for col in df_aux.columns:
        col_lower = col.lower()
        if any(kw in col_lower for kw in occ_keywords):
            occ_col = col
            print(f"   Occupancy column found: '{col}'")
            break
    if occ_col is None:
        print("   ⚠ No occupancy column detected. Check column names in the file.")

    # Procesar ocupación
    df_occ = None
    if occ_col:
        occ_series = pd.to_numeric(df_aux[occ_col], errors='coerce')
        if occ_series.isna().all():
            def text_to_occ(val):
                if pd.isna(val):
                    return np.nan
                s = str(val).strip().lower()
                return 1 if any(x in s for x in ['sí', 'si', 'yes', '1', 'true', 'ocupado', 'abierto']) else 0
            occ_series = df_aux[occ_col].apply(text_to_occ)

        # ✅ FIX 2: dropna primero, astype(int) después
        df_occ = df_aux[['DateTime']].copy()
        df_occ['Occupancy'] = occ_series
        df_occ = df_occ.dropna(subset=['Occupancy'])
        df_occ['Occupancy'] = df_occ['Occupancy'].astype(int)
        print(f"   Occupancy: {len(df_occ)} valid records")
        print(f"   Occupancy date range: {df_occ['DateTime'].min()} → {df_occ['DateTime'].max()}")

    # --- 3. Detectar columnas de puerta y ventana ---
    door_col = None
    window_col = None
    for col in df_aux.columns:
        col_lower = col.lower()
        if any(x in col_lower for x in ['door', 'puerta', 'porta']):
            door_col = col
        if any(x in col_lower for x in ['window', 'ventana', 'finestra']):
            window_col = col

    vent_data = []
    for _, row in df_aux.iterrows():
        door_val   = row[door_col]   if door_col   else None
        window_val = row[window_col] if window_col else None
        door_status, window_status = parse_detailed_ventilation(door_val, window_val)
        vent_data.append({
            'DateTime': row['DateTime'],
            'Door_Open': door_status,
            'Window_Open': window_status
        })

    df_vent = pd.DataFrame(vent_data)
    print(f"   Ventilation: {len(df_vent)} records")
    if not df_vent.empty:
        print(f"   Ventilation date range: {df_vent['DateTime'].min()} → {df_vent['DateTime'].max()}")

    return df_occ, df_vent

def process_sensors_to_pivot(df_period):
    """Transforma los datos del sensor de formato largo a formato pivotado,
    donde cada fila es una variable y cada columna es un timestamp real."""
    variables = [v for v in SENSOR_VARIABLES if v in df_period.columns]
    if not variables:
        return None
    df_long = pd.concat([
        df_period[['DateTime', 'Room', 'Volume', 'Floor', 'Sensor_Location', var]]
        .dropna(subset=[var])
        .rename(columns={var: 'Value'})
        .assign(Variable_Measure=var)
        for var in variables
    ])
    df_long = df_long.groupby(
        ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure', 'DateTime']
    )['Value'].mean().reset_index()
    df_pivot = df_long.set_index(
        ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure', 'DateTime']
    )['Value'].unstack(level='DateTime').reset_index()
    df_pivot['Variable_Measure'] = pd.Categorical(
        df_pivot['Variable_Measure'], categories=SENSOR_VARIABLES, ordered=True)
    return df_pivot.sort_values(['Sensor_Location', 'Variable_Measure'])

def add_outdoor_rows_to_pivot(df_pivot, df_outdoor, period_start, period_end):
    """Añade al pivote semanal las filas de variables exteriores."""
    if df_outdoor is None:
        return None
    df_period = df_outdoor[(df_outdoor['DateTime'] >= period_start) & (df_outdoor['DateTime'] <= period_end)]
    if len(df_period) == 0:
        return None

    outdoor_vars = ['Temperature_Outdoor', 'Humidity_Outdoor', 'Solar_Radiation']
    inc_names = {
        'Temperature_Outdoor': 'Temp_Outdoor_Increment',
        'Humidity_Outdoor':    'Humidity_Outdoor_Increment',
        'Solar_Radiation':     'Solar_Radiation_Increment'
    }

    rows = []
    for var in outdoor_vars:
        if var not in df_period.columns:
            continue
        row = {'Room': 1.6, 'Volume': 202, 'Floor': 'Ground',
               'Sensor_Location': 'Outdoor', 'Variable_Measure': var}
        inc_row = {'Room': 1.6, 'Volume': 202, 'Floor': 'Ground',
                   'Sensor_Location': 'Outdoor', 'Variable_Measure': inc_names[var]}
        for _, record in df_period.iterrows():
            if pd.notna(record[var]):
                row[record['DateTime']]     = record[var]
                inc_row[record['DateTime']] = np.nan
        rows.append(row)
        rows.append(inc_row)

    return pd.DataFrame(rows) if rows else None

def add_auxiliary_rows_to_pivot(df_pivot, df_occ, df_vent, period_start, period_end):
    """Añade al pivote semanal las filas de ocupación, puerta y ventana."""
    if df_occ is None and df_vent is None:
        return None
    metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']
    df_occ_period  = df_occ[(df_occ['DateTime'] >= period_start) & (df_occ['DateTime'] <= period_end)] if df_occ is not None else None
    df_vent_period = df_vent[(df_vent['DateTime'] >= period_start) & (df_vent['DateTime'] <= period_end)] if df_vent is not None else None
    if (df_occ_period is None or len(df_occ_period) == 0) and (df_vent_period is None or len(df_vent_period) == 0):
        return None

    existing_dates = [c for c in df_pivot.columns if c not in metadata_cols]
    aux_rows = []

    if df_occ_period is not None and len(df_occ_period) > 0:
        row = {'Room': 1.6, 'Volume': 202, 'Floor': 'Ground',
               'Sensor_Location': '', 'Variable_Measure': 'Occupancy'}
        for _, r in df_occ_period.iterrows():
            dt = r['DateTime']
            row[dt] = r['Occupancy']
            if dt not in existing_dates:
                existing_dates.append(dt)
        aux_rows.append(pd.DataFrame([row]))

    if df_vent_period is not None and len(df_vent_period) > 0:
        door_row   = {'Room': 1.6, 'Volume': 202, 'Floor': 'Ground', 'Sensor_Location': '', 'Variable_Measure': 'Door_Open'}
        window_row = {'Room': 1.6, 'Volume': 202, 'Floor': 'Ground', 'Sensor_Location': '', 'Variable_Measure': 'Window_Open'}
        for _, r in df_vent_period.iterrows():
            dt = r['DateTime']
            door_row[dt]   = r['Door_Open']
            window_row[dt] = r['Window_Open']
            if dt not in existing_dates:
                existing_dates.append(dt)
        aux_rows.append(pd.DataFrame([door_row]))
        aux_rows.append(pd.DataFrame([window_row]))

    if aux_rows:
        df_aux = pd.concat(aux_rows, ignore_index=True)
        for col in existing_dates:
            if col not in df_aux.columns:
                df_aux[col] = None
        date_cols = sorted([c for c in df_aux.columns if c not in metadata_cols])
        df_aux = df_aux[metadata_cols + date_cols]
        return df_aux
    return None

def save_sensor_to_excel(df, file_out, sheet_name):
    """Guarda el pivote semanal en Excel con formato de cabecera."""
    with pd.ExcelWriter(file_out, engine='openpyxl') as writer:
        df.to_excel(writer, sheet_name=sheet_name[:31], index=False)
        worksheet = writer.sheets[sheet_name[:31]]
        for cell in worksheet[1]:
            cell.fill = PatternFill(start_color='FFF2CC', end_color='FFF2CC', fill_type='solid')
            cell.font = Font(bold=True, size=11)
            cell.alignment = Alignment(horizontal='center')
        worksheet.freeze_panes = 'F2'
        for col_idx, col_name in enumerate(df.columns, 1):
            width = 18 if col_name in ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure'] else 20
            worksheet.column_dimensions[get_column_letter(col_idx)].width = width

def extract_occupancy_blocks(df_week, week_name, year, month):
    """Identifica bloques continuos de ocupación y calcula estadísticos por bloque."""
    metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']
    occ_row = df_week[df_week['Variable_Measure'] == 'Occupancy']
    if occ_row.empty:
        return []

    occ_row = occ_row.iloc[0]
    date_cols = sorted([c for c in df_week.columns if c not in metadata_cols])

    occupancy_sequence = []
    for col in date_cols:
        val = occ_row[col]
        if pd.notna(val):
            try:
                occupancy_sequence.append({
                    'DateTime': pd.to_datetime(col),
                    'Occupancy': int(val)
                })
            except:
                continue

    if not occupancy_sequence:
        return []

    blocks = []
    current_block = [occupancy_sequence[0]]
    for i in range(1, len(occupancy_sequence)):
        if occupancy_sequence[i]['Occupancy'] != occupancy_sequence[i-1]['Occupancy']:
            blocks.append(current_block)
            current_block = [occupancy_sequence[i]]
        else:
            current_block.append(occupancy_sequence[i])
    blocks.append(current_block)

    rows = []
    sensor_data = df_week[~df_week['Variable_Measure'].isin(['Occupancy', 'Door_Open', 'Window_Open'])]
    date_cols_dt = [pd.to_datetime(c) for c in date_cols]

    for block in blocks:
        block_start = block[0]['DateTime']
        block_end = block[-1]['DateTime']
        block_indices = [i for i, dt in enumerate(date_cols_dt) if block_start <= dt <= block_end]

        if not block_indices:
            continue

        row = {
            'Occupancy': block[0]['Occupancy'],
            'Date': block_start.date(),
            'Day_of_Week': block_start.strftime('%A'),
            'Start_Time': block_start.time(),
            'End_Time': block_end.time(),
            'Duration_min': round((block_end - block_start).total_seconds() / 60),
            'Week': week_name
        }

        for _, sensor_row in sensor_data.iterrows():
            values = [float(sensor_row[date_cols[idx]]) for idx in block_indices
                     if pd.notna(sensor_row[date_cols[idx]])]
            if values:
                prefix = f"{sensor_row['Sensor_Location']}_{sensor_row['Variable_Measure']}" if sensor_row['Sensor_Location'] else sensor_row['Variable_Measure']
                row.update({
                    f'{prefix}_mean': np.mean(values),
                    f'{prefix}_std': np.std(values) if len(values) > 1 else 0,
                    f'{prefix}_max': np.max(values),
                    f'{prefix}_min': np.min(values)
                })
        rows.append(row)

    return rows


# ==================== PROGRAMA PRINCIPAL ====================

# 1. Cargar datos outdoor
print("Loading Outdoor data...")
df_outdoor_raw = load_outdoor_data()
if df_outdoor_raw is None:
    raise SystemExit("File 'Outdoor parameters. SIAR.xlsx' not found or could not be loaded.")

print(f"   Loaded: {len(df_outdoor_raw)} records")
df_outdoor_filtered = filter_weekdays_time(df_outdoor_raw)
print(f"   After filter (Mon-Fri, 6:00-23:30): {len(df_outdoor_filtered)} records")
print(f"   Date range: {df_outdoor_filtered['DateTime'].min()} → {df_outdoor_filtered['DateTime'].max()}")

# ==================== CLASSROOM 1.6 FILES ====================
print("\n" + "="*50)
print("Processing Classroom 1.6 data...")
print("="*50)

# 2. Cargar datos de sensores
all_dfs = []
for file_in, room_id, sensor_loc in sensor_files:
    df = load_dataframe(file_in)
    if df is None:
        continue
    rename_map = {}
    for col in df.columns:
        col_lower = str(col).lower()
        if 'temp' in col_lower and 'increment' not in col_lower:
            rename_map[col] = 'Temperature'
        elif 'hum' in col_lower and 'co2' not in col_lower:
            rename_map[col] = 'Humidity'
        elif 'co2' in col_lower:
            rename_map[col] = 'CO2'
    df = df.rename(columns=rename_map)
    df['Room'], df['Volume'], df['Floor'], df['Sensor_Location'] = room_id, 202, 'Ground', sensor_loc
    all_dfs.append(df)

if not all_dfs:
    raise SystemExit("No sensor data found")

df_all = pd.concat(all_dfs, ignore_index=True).sort_values(['Sensor_Location', 'DateTime'])

# 3. Cargar datos complementarios
df_occ, df_vent = load_auxiliary_data('1.6 occupancy.xlsx')

# 4. Filtrar datos (lunes-viernes, 6:00-23:30)
print("Filtering data (Mon-Fri, 6:00-23:30)...")
total_before = len(df_all)
df_all = filter_weekdays_time(df_all)
print(f"   Sensor data: {total_before} → {len(df_all)} records ({(1 - len(df_all)/total_before)*100:.1f}% removed)")

df_outdoor = df_outdoor_filtered.drop(columns=['Year', 'Month'], errors='ignore')
df_occ  = filter_weekdays_time(df_occ)
df_vent = filter_weekdays_time(df_vent)

for name, df in [("Outdoor", df_outdoor), ("Occupancy", df_occ), ("Ventilation", df_vent)]:
    if df is not None:
        print(f"   {name}: {len(df)} records after filter")

# 5. Inicializar columnas de incremento a 0
for base_col, inc_col in [('Temperature', 'Temp_Increment'), ('Humidity', 'Humidity_Increment'), ('CO2', 'CO2_Increment')]:
    if base_col in df_all.columns:
        df_all[inc_col] = df_all[base_col].notna().astype(float) * 0

# 6. Procesar por semanas
metadata_cols = ['Room', 'Volume', 'Floor', 'Sensor_Location', 'Variable_Measure']
df_all['Year'], df_all['Month'] = df_all['DateTime'].dt.year, df_all['DateTime'].dt.month
generated_sensor_files = []
all_occupancy_rows = []

for (year, month) in df_all[['Year', 'Month']].drop_duplicates().sort_values(['Year', 'Month']).values:
    month_name = datetime(year, month, 1).strftime('%B')
    print(f"\n{month_name} {year}")

    df_month = df_all[(df_all['Year'] == year) & (df_all['Month'] == month)]

    for week_name, week_data, start, end in split_into_weeks(df_month, year, month):
        print(f"\n   Processing {week_name}...")
        df_pivot = process_sensors_to_pivot(week_data)
        if df_pivot is None:
            print(f"      No sensor data for this week")
            continue

        outdoor_rows = add_outdoor_rows_to_pivot(df_pivot, df_outdoor, start, end)
        aux_rows = add_auxiliary_rows_to_pivot(df_pivot, df_occ, df_vent, start, end)

        # Concatenar todos los bloques
        df_parts = [df_pivot]
        if outdoor_rows is not None:
            df_parts.append(outdoor_rows)
        if aux_rows is not None:
            df_parts.append(aux_rows)
        df_final = pd.concat(df_parts, ignore_index=True)

        # ✅ FIX FFILL GLOBAL:
        # Recalcular all_dates DESPUÉS del concat para capturar todos los
        # timestamps reales de sensores, exterior y ocupación juntos.
        # Ordenar cronológicamente antes del ffill para que la propagación
        # hacia adelante sea correcta independientemente del intervalo.
        all_dates = sorted([c for c in df_final.columns if c not in metadata_cols])

        # Variables continuas: ffill + bfill (bfill solo cubre el primer NaN)
        continuous_mask = df_final['Variable_Measure'].isin([
            'Temperature', 'Humidity', 'CO2',
            'Temperature_Outdoor', 'Humidity_Outdoor', 'Solar_Radiation'
        ])
        if continuous_mask.any():
            df_final.loc[continuous_mask, all_dates] = (
                df_final.loc[continuous_mask, all_dates]
                .astype(float)
                .ffill(axis=1)
                .bfill(axis=1)
            )

        # Variables auxiliares: solo ffill (no inventar ocupación antes del primer dato)
        aux_mask = df_final['Variable_Measure'].isin(['Occupancy', 'Door_Open', 'Window_Open'])
        if aux_mask.any():
            df_final.loc[aux_mask, all_dates] = (
                df_final.loc[aux_mask, all_dates]
                .astype(float)
                .ffill(axis=1)
            )

        # Reordenar columnas: metadatos primero, luego fechas cronológicas
        df_final = df_final[metadata_cols + all_dates]

        # Calcular incrementos
        pairs = {
            'Temperature': 'Temp_Increment',
            'Humidity': 'Humidity_Increment',
            'CO2': 'CO2_Increment',
            'Temperature_Outdoor': 'Temp_Outdoor_Increment',
            'Humidity_Outdoor': 'Humidity_Outdoor_Increment',
            'Solar_Radiation': 'Solar_Radiation_Increment'
        }

        outdoor_base_vars = {'Temperature_Outdoor', 'Humidity_Outdoor', 'Solar_Radiation'}

        for base_var, inc_var in pairs.items():
            base_rows = df_final[df_final['Variable_Measure'] == base_var]
            inc_rows  = df_final[df_final['Variable_Measure'] == inc_var]

            if base_rows.empty or inc_rows.empty:
                continue

            for loc in base_rows['Sensor_Location'].unique():
                base_idx = base_rows[base_rows['Sensor_Location'] == loc].index
                inc_idx  = inc_rows[inc_rows['Sensor_Location'] == loc].index

                if base_idx.empty or inc_idx.empty:
                    continue

                existing_dates = [c for c in all_dates if c in df_final.columns]

                if base_var in outdoor_base_vars:
                    df_period_out = df_outdoor[
                        (df_outdoor['DateTime'] >= start) & (df_outdoor['DateTime'] <= end)
                    ].copy()

                    if base_var in df_period_out.columns:
                        df_period_out = df_period_out.set_index('DateTime')[base_var].dropna().sort_index()

                        if not df_period_out.empty:
                            base_values = {}
                            for date_col in existing_dates:
                                if date_col in df_period_out.index:
                                    base_values[date_col] = df_period_out[date_col]
                                else:
                                    time_diffs = abs(df_period_out.index - date_col)
                                    min_diff = time_diffs.min()
                                    if min_diff <= pd.Timedelta('30minutes'):
                                        closest_idx = df_period_out.index[time_diffs.argmin()]
                                        base_values[date_col] = df_period_out[closest_idx]

                            if base_values:
                                sorted_dates = sorted(base_values.keys())
                                for date_col, value in base_values.items():
                                    df_final.loc[base_idx, date_col] = value
                                for i, date_col in enumerate(sorted_dates):
                                    if i == 0:
                                        df_final.loc[inc_idx, date_col] = np.nan
                                    else:
                                        prev_date    = sorted_dates[i - 1]
                                        current_val  = base_values[date_col]
                                        prev_val     = base_values[prev_date]
                                        hours_elapsed = (date_col - prev_date).total_seconds() / 3600
                                        if hours_elapsed > 0:
                                            df_final.loc[inc_idx, date_col] = current_val - prev_val
                                        else:
                                            df_final.loc[inc_idx, date_col] = np.nan
                                other_dates = [c for c in existing_dates if c not in base_values]
                                df_final.loc[inc_idx, other_dates] = np.nan
                                df_final.loc[base_idx, other_dates] = np.nan
                else:
                    if not base_idx.empty:
                        base_series = df_final.loc[base_idx, existing_dates].astype(float).iloc[0]
                        real_dates  = [c for c in existing_dates if pd.notna(base_series[c])]

                        increments = [np.nan]
                        for i in range(1, len(real_dates)):
                            curr_val      = base_series[real_dates[i]]
                            prev_val      = base_series[real_dates[i - 1]]
                            hours_elapsed = (real_dates[i] - real_dates[i - 1]).total_seconds() / 3600
                            if hours_elapsed > 0:
                                increments.append(curr_val - prev_val)
                            else:
                                increments.append(np.nan)

                        df_final.loc[inc_idx, real_dates] = increments
                        other_dates = [c for c in existing_dates if c not in real_dates]
                        df_final.loc[inc_idx, other_dates] = np.nan

            # Eliminar filas de incremento sin fila base correspondiente
            locs_with_base = set(base_rows['Sensor_Location'].unique())
            for loc in inc_rows['Sensor_Location'].unique():
                if loc not in locs_with_base:
                    drop_idx = inc_rows[inc_rows['Sensor_Location'] == loc].index
                    df_final = df_final.drop(index=drop_idx)

        # Guardar Excel semanal
        out_file = f'1.6_data_{month_name}_{year}_{week_name}.xlsx'
        save_sensor_to_excel(df_final, out_file, f'{month_name}_{week_name}')
        generated_sensor_files.append(out_file)
        print(f"   Saved: {out_file}")

        # Extraer bloques de ocupación
        occupancy_rows = extract_occupancy_blocks(df_final, week_name, year, month)
        if occupancy_rows:
            all_occupancy_rows.extend(occupancy_rows)
            print(f"   Occupancy blocks found: {len(occupancy_rows)}")

# 7. Guardar resumen de ocupación
if all_occupancy_rows:
    df_occupancy_summary = pd.DataFrame(all_occupancy_rows)
    occupancy_file = '1.6_occupancy_blocks_summary.xlsx'
    save_to_excel(df_occupancy_summary, occupancy_file, 'Occupancy_Blocks')
    print(f"\nOccupancy blocks summary saved: {occupancy_file}")
    print(f"Total occupancy blocks: {len(all_occupancy_rows)}")

print("\n" + "="*50)
print("Processing completed successfully!")
print("="*50)
print(f"\nGenerated files for Classroom 1.6:")
for file in generated_sensor_files:
    print(f"  - {file}")
if all_occupancy_rows:
    print(f"  - {occupancy_file}")

Loading Outdoor data...
   Loaded: 4512 records
   After filter (Mon-Fri, 6:00-23:30): 2376 records
   Date range: 2025-12-19 06:00:00 → 2026-03-20 23:30:00

Processing Classroom 1.6 data...
   Loaded: 82 rows
   DateTime range after parse: 2026-02-09 10:17:00 → 2026-03-18 11:15:00
   Occupancy column found: 'Occupancy'
   Occupancy: 78 valid records
   Occupancy date range: 2026-02-09 10:17:00 → 2026-03-18 11:15:00
   Ventilation: 82 records
   Ventilation date range: 2026-02-09 10:17:00 → 2026-03-18 11:15:00
Filtering data (Mon-Fri, 6:00-23:30)...
   Sensor data: 39225 → 21048 records (46.3% removed)
   Outdoor: 2376 records after filter
   Occupancy: 78 records after filter
   Ventilation: 82 records after filter

December 2025

   Processing week_4_days_22-31...
   Saved: 1.6_data_December_2025_week_4_days_22-31.xlsx

January 2026

   Processing week_1_days_1-7...
   Saved: 1.6_data_January_2026_week_1_days_1-7.xlsx

   Processing week_2_days_8-14...
   Saved: 1.6_data_January_2026